In [ ]:
!pip install torch transformers datasets accelerate bitsandbytes peft unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, Trainer
# from peft import LoraConfig
from transformers import DataCollatorForLanguageModeling
import os

# Disable Weights & Biases logging
os.environ["WANDB_DISABLED"] = "true"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# 1. Load LLaMA-3.2 1B in 4-bit mode
model_name = "meta-llama/Llama-3.2-1b"  # Adjust to actual model name
max_seq_length = 512  # Maximum sequence length

# Load model in 4-bit quantized mode
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name,
    load_in_4bit=True,
    max_seq_length=max_seq_length
)

# 2. Attach LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=8,  # LoRA rank (controls trainable parameter count)
    lora_alpha=32,  # Scaling factor for LoRA
    lora_dropout=0.05,  # Dropout rate to prevent overfitting
    bias="none",  # No bias training
    use_rslora=False,  # Keep standard LoRA
)



==((====))==  Unsloth 2025.2.15: Fast Llama patching. Transformers: 4.48.3.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.2.15 patched 16 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [ ]:
# Print trainable parameters
model.print_trainable_parameters()

trainable params: 5,636,096 || all params: 1,241,450,496 || trainable%: 0.4540


In [ ]:
# 3. Load and Tokenize Datasets (WikiText + BooksCorpus)
dataset_wiki = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset_books = load_dataset("bookcorpus")  # BookCorpus dataset


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/18.5k [00:00<?, ?B/s]

bookcorpus.py:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

The repository for bookcorpus contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bookcorpus.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/74004228 [00:00<?, ? examples/s]

KeyError: 'validation'

In [ ]:
split_ratio = 0.98  # Use 98% of BooksCorpus for training, 2% for validation
books_train_test = dataset_books["train"].train_test_split(test_size=1 - split_ratio, seed=42)

# Assign new train/validation splits
dataset_books = {
    "train": books_train_test["train"].select(range(1000)),  # Select only 1,000 samples
    "validation": books_train_test["test"].select(range(100))  # Select 100 samples
}

# Reduce WikiText dataset size
dataset_wiki = {
    "train": dataset_wiki["train"].select(range(1000)),  # Select 1,000 samples
    "validation": dataset_wiki["validation"].select(range(100))  # Select 100 samples
}

# Merge datasets after reducing size
dataset_combined = {
    "train": concatenate_datasets([dataset_wiki["train"], dataset_books["train"]]),
    "validation": concatenate_datasets([dataset_wiki["validation"], dataset_books["validation"]]),
}
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_seq_length)


In [ ]:
# Apply tokenization
tokenized_datasets = {split: dataset.map(tokenize_function, batched=True) for split, dataset in dataset_combined.items()}

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
small_train_dataset = tokenized_datasets["train"]
small_eval_dataset = tokenized_datasets["validation"]

In [ ]:
# 4. Training Setup
training_args = TrainingArguments(
    output_dir="./llama-unsloth",
    per_device_train_batch_size=16,  # Reduced for memory efficiency
    per_device_eval_batch_size=4,
    learning_rate=1.5e-4,  # Adjusted for stability
    num_train_epochs=3,  # More epochs for better training
    save_strategy="epoch",
    optim="adamw_torch",
    fp16=True,  # Mixed precision training
    logging_steps=5,  # More frequent logging
)

# Data collator for Causal LM training
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Set to False for autoregressive models like LLaMA
)

# 5. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    data_collator=data_collator,  # Handles labels correctly
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [ ]:
# 6. Train the Model
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 16 | Gradient Accumulation steps = 1
\        /    Total batch size = 16 | Total steps = 375
 "-____-"     Number of trainable parameters = 5,636,096


Step,Training Loss
5,3.372600
10,3.254000
15,3.379300
20,3.488400
25,3.085100
30,3.051300
35,3.027000
40,2.978200
45,2.779900
50,3.353700


TrainOutput(global_step=375, training_loss=2.5073958384195962, metrics={'train_runtime': 1276.5649, 'train_samples_per_second': 4.7, 'train_steps_per_second': 0.294, 'total_flos': 1.8040913657856e+16, 'train_loss': 2.5073958384195962, 'epoch': 3.0})

In [ ]:
# 7. Inference: Answer Questions
questions = [
    "If it takes 1 hour for 60 people to play an opera, how many hours will it take 600 people to play the same opera?",
    "Is a pound of feathers or a British pound heavier?",
    "A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under the tree. What's going on?",
    "What happens if you crack your knuckles a lot?",
    "If there is a shark in the pool of my basement, is it safe to go upstairs?",
    "How much wood could a woodchuck chuck if there were only 5 pounds of wood in the world?",
    "Who is the current President of the United States?",
    "Was Talos alive?",
    "How many Ls are in the word parallel?",
    "What is the riddle of the sphinx, and what are all possible answers satisfying all conditions?"
]



In [ ]:
# Enable inference mode for Unsloth models
FastLanguageModel.for_inference(model)

def generate_answer(question):
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        output = model.generate(**inputs, max_length=100, pad_token_id=tokenizer.eos_token_id)

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    # Remove the original prompt from the generated text
    answer = response.replace(prompt, "").strip()

    return answer



In [ ]:
print("\n--- Model Responses ---\n")
for q in questions:
    print(f"Q: {q}\nA: {generate_answer(q)}\n")


--- Model Responses ---

Q: If it takes 1 hour for 60 people to play an opera, how many hours will it take 600 people to play the same opera?
A: 120 hours 
Explanation: 
The number of hours required to play the opera is the same as the number of hours taken by 600 people to play the same opera. 
This means that the number of hours required is the same as the length of a single day. 
The fact that the number of hours required is

Q: Is a pound of feathers or a British pound heavier?
A: British pound is heavier than a pound of feathers. 
Answer: British pound is lighter than a pound of feathers. 
Answer: British pound is heavier than a pound of feathers 
Answer: British pound is heavier than a pound of feathers 
Answer: British pound is lighter than a pound of feathers 
Answer: British pound is lighter than a pound of feathers 
Answer: British pound is heavier than a pound of feathers

Q: A boy runs down the stairs in the morning and sees a tree in his living room, and some boxes under 